# Qwen3 1.7B Unsloth Fine-Tuning + GGUF Export

This notebook follows the exact workflow you asked for:
1. Download base model in Colab
2. Train LoRA adapters with QLoRA using Unsloth
3. Merge and export GGUF for llama.cpp or Ollama

Important reality check: CPU usage under a hard 20 percent cannot be guaranteed on every local rig.
This notebook includes export and launch settings that maximize the chance of staying near that target on GTX 1060 3GB.

In [ ]:
!pip -q install -U unsloth trl datasets accelerate bitsandbytes sentencepiece huggingface_hub

In [ ]:
import os
import glob
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Mount Drive and Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/NoteBookPRO/training_data/qwen_golden_dataset.jsonl'
OUT_DIR = '/content/qwen3_unsloth_run'
ADAPTER_DIR = f'{OUT_DIR}/adapter'
MERGED_DIR = f'{OUT_DIR}/merged_16bit'
GGUF_Q4_DIR = f'{OUT_DIR}/gguf_q4'
GGUF_Q3_DIR = f'{OUT_DIR}/gguf_q3'
os.makedirs(OUT_DIR, exist_ok=True)

print('Dataset path:', DATA_PATH)
print('Exists:', os.path.exists(DATA_PATH))

## Training Config

In [ ]:
MAX_SEQ_LEN = 2048
EPOCHS = 2
LR = 1e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8
VAL_SPLIT = 0.02
SEED = 42

# Data quality guards for fine-tune stability.
MIN_ASSISTANT_CHARS = 180
REQUIRE_HEADINGS = True
REQUIRE_CITATIONS = True

MODEL_CANDIDATES = [
    'unsloth/Qwen3-1.7B-bnb-4bit',
    'Qwen/Qwen3-1.7B',
]

## Load Model with Unsloth

In [ ]:
model = None
tokenizer = None
chosen_model = None

for model_name in MODEL_CANDIDATES:
    try:
        print('Trying:', model_name)
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=MAX_SEQ_LEN,
            dtype=None,
            load_in_4bit=True,
        )
        chosen_model = model_name
        break
    except Exception as e:
        print('Failed:', model_name, '->', str(e)[:300])

if model is None:
    raise RuntimeError('Could not load any Qwen3 model candidate in Unsloth.')

print('Using model:', chosen_model)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

## Load and Normalize Dataset

In [ ]:
import hashlib
import re

IM_START = '<|im_start|>'
IM_END = '<|im_end|>'

BAD_ASSISTANT_PATTERNS = [
    r'<think>[\s\S]*?</think>',
    r'<\|begin_of_thought\|>[\s\S]*?<\|end_of_thought\|>',
    r'only output the final answer in markdown',
    r'answer must be in markdown format only',
    r'keep the answer strictly within the markdown block',
    r'formatting policy for this answer',
    r'question\s*:\s*',
    r'partial answer\s*:\s*',
]

def parse_qwen_chatml(text: str):
    text = text.replace('\r\n', '\n').replace('\r', '\n').strip()
    roles = {}
    for role in ('system', 'user', 'assistant'):
        marker = f'{IM_START}{role}\n'
        i = text.find(marker)
        if i < 0:
            return None
        i += len(marker)
        j = text.find(IM_END, i)
        if j < 0:
            return None
        roles[role] = text[i:j].strip()
    if not all(roles.values()):
        return None
    return [
        {'role': 'system', 'content': roles['system']},
        {'role': 'user', 'content': roles['user']},
        {'role': 'assistant', 'content': roles['assistant']},
    ]

def normalize_messages(msgs):
    if not isinstance(msgs, list):
        return None

    by_role = {}
    for item in msgs:
        if not isinstance(item, dict):
            continue
        role = str(item.get('role', '')).strip().lower()
        content = str(item.get('content', '')).strip()
        if role in ('system', 'user', 'assistant') and content and role not in by_role:
            by_role[role] = content

    if not all(r in by_role for r in ('system', 'user', 'assistant')):
        return None

    return [
        {'role': 'system', 'content': by_role['system']},
        {'role': 'user', 'content': by_role['user']},
        {'role': 'assistant', 'content': by_role['assistant']},
    ]

def clean_assistant(answer: str) -> str:
    if not answer:
        return ''
    cleaned = answer
    cleaned = re.sub(r'<think>[\s\S]*?</think>', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'<\|begin_of_thought\|>[\s\S]*?<\|end_of_thought\|>', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^\s*only output the final answer in markdown\.?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^\s*answer must be in markdown format only\.?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^\s*keep the answer strictly within the markdown block\.?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^\s*answer\s*:\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned).strip()
    return cleaned

def looks_bad_assistant(answer: str) -> bool:
    if not answer or len(answer) < MIN_ASSISTANT_CHARS:
        return True

    low = answer.lower()
    if REQUIRE_HEADINGS and '### ' not in answer:
        return True
    if REQUIRE_CITATIONS and not re.search(r'\[(?:\d+\s*(?:,\s*\d+)*)\]', answer):
        return True

    # Reject rows that still carry obvious prompt/meta leakage.
    leak_hits = 0
    for pat in BAD_ASSISTANT_PATTERNS:
        if re.search(pat, low, flags=re.IGNORECASE):
            leak_hits += 1
    if leak_hits >= 2:
        return True

    # Repetition guard: same sentence repeated >= 3 times.
    sentences = [s.strip().lower() for s in re.split(r'(?<=[.!?])\s+', answer) if len(s.strip()) > 20]
    if len(sentences) >= 6:
        counts = {}
        for s in sentences:
            counts[s] = counts.get(s, 0) + 1
        if max(counts.values()) >= 3:
            return True

    return False

def to_messages(example):
    msgs = example.get('messages')
    if isinstance(msgs, list) and len(msgs) >= 3:
        parsed = normalize_messages(msgs)
    else:
        parsed = None

    if parsed is None:
        txt = example.get('text')
        if isinstance(txt, str):
            parsed = parse_qwen_chatml(txt)

    if parsed is None:
        return {'messages': None, 'assistant_fp': None}

    assistant = clean_assistant(parsed[2]['content'])
    if looks_bad_assistant(assistant):
        return {'messages': None, 'assistant_fp': None}

    parsed[2]['content'] = assistant
    fp = hashlib.sha256(re.sub(r'\s+', ' ', assistant.lower()).encode('utf-8')).hexdigest()
    return {'messages': parsed, 'assistant_fp': fp}

raw = load_dataset('json', data_files=DATA_PATH, split='train')
raw_count = len(raw)

norm = raw.map(to_messages)
norm = norm.filter(lambda x: x['messages'] is not None)

# Near-exact dedupe on normalized assistant answer.
seen = set()
def dedupe_keep_first(example):
    fp = example.get('assistant_fp')
    if not fp or fp in seen:
        return False
    seen.add(fp)
    return True

norm = norm.filter(dedupe_keep_first)
norm = norm.remove_columns(['assistant_fp'])

print('Raw samples:', raw_count)
print('Usable samples after sanitization/dedupe:', len(norm))

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def apply_chat_template_no_think(messages):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
    except TypeError:
        # Fallback for tokenizer versions that do not expose enable_thinking.
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

def format_for_train(example):
    text = apply_chat_template_no_think(example['messages'])
    return {'text': text + tokenizer.eos_token}

dataset = norm.map(format_for_train, remove_columns=norm.column_names)
split = dataset.train_test_split(test_size=VAL_SPLIT, seed=SEED)

print(split)
print(split['train'][0]['text'][:700])

## Train with SFTTrainer

In [ ]:
import inspect
from dataclasses import dataclass
from transformers import EarlyStoppingCallback, TrainingArguments
from trl import SFTTrainer

ASSISTANT_PREFIX = "<|im_start|>assistant\n"
ASSISTANT_END = "<|im_end|>"
assistant_prefix_ids = tokenizer.encode(ASSISTANT_PREFIX, add_special_tokens=False)
assistant_end_ids = tokenizer.encode(ASSISTANT_END, add_special_tokens=False)

if not assistant_prefix_ids:
    raise RuntimeError("Failed to tokenize assistant prefix marker.")
if not assistant_end_ids:
    raise RuntimeError("Failed to tokenize assistant end marker.")

def find_subsequence(seq, subseq, start_idx=0):
    if not subseq:
        return -1
    max_i = len(seq) - len(subseq) + 1
    for i in range(max(start_idx, 0), max_i):
        if seq[i:i + len(subseq)] == subseq:
            return i
    return -1

def build_assistant_only_labels(input_ids):
    labels = [-100] * len(input_ids)

    start_idx = -1
    for i in range(0, len(input_ids) - len(assistant_prefix_ids) + 1):
        if input_ids[i:i + len(assistant_prefix_ids)] == assistant_prefix_ids:
            start_idx = i + len(assistant_prefix_ids)

    if start_idx < 0:
        return labels

    end_marker_idx = find_subsequence(input_ids, assistant_end_ids, start_idx=start_idx)
    end_idx = end_marker_idx if end_marker_idx >= 0 else len(input_ids)

    for j in range(start_idx, end_idx):
        labels[j] = input_ids[j]

    return labels

def tokenize_batch(batch):
    tok = tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    tok['labels'] = [build_assistant_only_labels(ids) for ids in tok['input_ids']]
    return tok

tokenized_train = split['train'].map(
    tokenize_batch,
    batched=True,
    remove_columns=split['train'].column_names,
    desc='Tokenizing train dataset (assistant-only labels)',
)
tokenized_eval = split['test'].map(
    tokenize_batch,
    batched=True,
    remove_columns=split['test'].column_names,
    desc='Tokenizing eval dataset (assistant-only labels)',
)

@dataclass
class AssistantOnlyCollator:
    tokenizer: object
    pad_to_multiple_of: int | None = 8

    def __call__(self, features):
        max_len = max(len(f['input_ids']) for f in features)
        if self.pad_to_multiple_of and max_len % self.pad_to_multiple_of != 0:
            max_len = ((max_len // self.pad_to_multiple_of) + 1) * self.pad_to_multiple_of

        input_ids_batch = []
        attention_batch = []
        labels_batch = []

        for f in features:
            ids = f['input_ids']
            attn = f.get('attention_mask', [1] * len(ids))
            labels = f['labels']

            pad_len = max_len - len(ids)
            input_ids_batch.append(ids + [tokenizer.pad_token_id] * pad_len)
            attention_batch.append(attn + [0] * pad_len)
            labels_batch.append(labels + [-100] * pad_len)

        return {
            'input_ids': torch.tensor(input_ids_batch, dtype=torch.long),
            'attention_mask': torch.tensor(attention_batch, dtype=torch.long),
            'labels': torch.tensor(labels_batch, dtype=torch.long),
        }

ta_sig = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
ta_kwargs = {
    'output_dir': OUT_DIR,
    'num_train_epochs': EPOCHS,
    'learning_rate': LR,
    'per_device_train_batch_size': BATCH_SIZE,
    'per_device_eval_batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM,
    'eval_steps': 50,
    'logging_steps': 10,
    'save_steps': 100,
    'save_total_limit': 2,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.01,
    'optim': 'paged_adamw_8bit',
    'fp16': not torch.cuda.is_bf16_supported(),
    'bf16': torch.cuda.is_bf16_supported(),
    'gradient_checkpointing': True,
    'report_to': 'none',
    'seed': SEED,
    'dataloader_num_workers': 0,
}

if 'evaluation_strategy' in ta_sig:
    ta_kwargs['evaluation_strategy'] = 'steps'
elif 'eval_strategy' in ta_sig:
    ta_kwargs['eval_strategy'] = 'steps'

if 'warmup_steps' in ta_sig:
    ta_kwargs['warmup_steps'] = 20
elif 'warmup_ratio' in ta_sig:
    ta_kwargs['warmup_ratio'] = 0.03

if 'load_best_model_at_end' in ta_sig:
    ta_kwargs['load_best_model_at_end'] = True
if 'metric_for_best_model' in ta_sig:
    ta_kwargs['metric_for_best_model'] = 'eval_loss'
if 'greater_is_better' in ta_sig:
    ta_kwargs['greater_is_better'] = False

training_args = TrainingArguments(**ta_kwargs)

data_collator = AssistantOnlyCollator(tokenizer=tokenizer)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Using assistant-only loss masking (system/user tokens ignored).')
print('Assistant text is truncated at <|im_end|> for labels to prevent prompt bleed.')
train_result = trainer.train()
print(train_result)

## Save Adapter, Merge, and Export GGUF

In [ ]:
import glob
import os
import shutil
import time
import traceback

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_Q4_DIR, exist_ok=True)
os.makedirs(GGUF_Q3_DIR, exist_ok=True)

total, used, free = shutil.disk_usage('/')
print(f"Disk free before export: {free / (1024**3):.2f} GB")

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Adapter saved:', ADAPTER_DIR)

model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print('Merged model saved:', MERGED_DIR)


def all_gguf_files_under_outdir():
    return set(glob.glob(f"{OUT_DIR}/**/*.gguf", recursive=True))


def export_gguf_checked(out_dir, quant_method):
    print(f"\nExporting {quant_method} -> {out_dir}")
    t0 = time.time()
    before = all_gguf_files_under_outdir()

    try:
        model.save_pretrained_gguf(out_dir, tokenizer, quantization_method=quant_method)
    except Exception as e:
        print(f"GGUF export failed for {quant_method}: {e}")
        traceback.print_exc()
        return []

    after = all_gguf_files_under_outdir()
    new_files = sorted(after - before)

    if new_files:
        print(f"Detected {len(new_files)} newly created GGUF file(s):")
        for fp in new_files:
            size_gb = os.path.getsize(fp) / (1024**3)
            print(f"Created: {fp} ({size_gb:.2f} GB)")
    else:
        # Fallback: some exporters may write into a suffixed sibling directory.
        candidates = sorted(
            set(glob.glob(f"{out_dir}/**/*.gguf", recursive=True))
            | set(glob.glob(f"{out_dir}_gguf/**/*.gguf", recursive=True))
        )
        if not candidates:
            print(f"No .gguf files found after {quant_method} export.")
        else:
            print(f"No delta detected, but found existing GGUF in candidate paths for {quant_method}:")
            for fp in candidates:
                size_gb = os.path.getsize(fp) / (1024**3)
                print(f"Found: {fp} ({size_gb:.2f} GB)")
            new_files = candidates

    print(f"Export elapsed: {time.time() - t0:.1f}s")
    return new_files


q4_files = export_gguf_checked(GGUF_Q4_DIR, 'q4_k_m')
q3_files = export_gguf_checked(GGUF_Q3_DIR, 'q3_k_m')

all_files = sorted(all_gguf_files_under_outdir())
if not all_files:
    print('\nNo GGUF files were generated. Re-run this cell and inspect traceback above.')
    print('Common causes: low disk space, missing conversion dependency, or failed merge export.')
else:
    print('\nGGUF export step completed. Total GGUF under OUT_DIR:', len(all_files))

In [ ]:
from pathlib import Path
import glob
import os
import re

all_files = sorted(glob.glob(f"{OUT_DIR}/**/*.gguf", recursive=True))

if not all_files:
    raise FileNotFoundError('No GGUF files found under OUT_DIR. Re-run the export cell and inspect its traceback output.')

q4_files = [fp for fp in all_files if re.search(r'(?i)q4', Path(fp).name)]
q3_files = [fp for fp in all_files if re.search(r'(?i)q3', Path(fp).name)]

print(f"Found total GGUF files under {OUT_DIR}: {len(all_files)}")
for fp in all_files:
    size_gb = os.path.getsize(fp) / (1024**3)
    print(f"  {fp} ({size_gb:.2f} GB)")

print(f"\nClassified Q4 files: {len(q4_files)}")
print(f"Classified Q3 files: {len(q3_files)}")

if not (q4_files or q3_files):
    print('\nNote: GGUF files exist but names do not contain Q3/Q4; download cell will still include all GGUF files.')

## Download GGUF to Your PC

In [ ]:
from pathlib import Path
import glob
import os
import time

all_files = sorted(glob.glob(f"{OUT_DIR}/**/*.gguf", recursive=True))

if not all_files:
    raise FileNotFoundError('No GGUF files to download. Run export cell first.')

print(f"Total GGUF files discovered under {OUT_DIR}: {len(all_files)}")
for fp in all_files:
    print(f"Found: {fp} ({os.path.getsize(fp) / (1024**3):.2f} GB)")

# Choose download mode:
# - 'direct' triggers browser download from Colab (no reliable progress UI from files.download).
# - 'drive' copies to Google Drive with progress logs, then you download from Drive with native progress.
DOWNLOAD_MODE = 'drive'  # change to 'direct' if you still want immediate browser download

if DOWNLOAD_MODE == 'direct':
    from google.colab import files

    print('Starting direct browser downloads...')
    print('Note: google.colab.files.download does not provide reliable progress bars for large files.')
    for fp in all_files:
        print(f"Triggering download: {Path(fp).name}")
        files.download(fp)
else:
    from google.colab import drive

    DRIVE_ROOT = '/content/drive/MyDrive/gguf_exports'
    drive.mount('/content/drive')

    timestamp = time.strftime('%Y%m%d_%H%M%S')
    target_dir = Path(DRIVE_ROOT) / f'qwen3_gguf_{timestamp}'
    target_dir.mkdir(parents=True, exist_ok=True)

    def copy_with_progress(src, dst, chunk_mb=16):
        total = os.path.getsize(src)
        copied = 0
        last_pct = -1
        chunk = chunk_mb * 1024 * 1024
        with open(src, 'rb') as rf, open(dst, 'wb') as wf:
            while True:
                data = rf.read(chunk)
                if not data:
                    break
                wf.write(data)
                copied += len(data)
                pct = int(copied * 100 / total)
                if pct != last_pct and pct % 5 == 0:
                    print(f"  {Path(src).name}: {pct}%")
                    last_pct = pct

    for src in all_files:
        dst = str(target_dir / Path(src).name)
        print(f"\nCopying to Drive: {src} -> {dst}")
        copy_with_progress(src, dst)
        print('  Done')

    print(f"\nAll files copied to: {target_dir}")
    print('Now download from Google Drive on your PC (you will get normal progress UI there).')

## Local GTX 1060 3GB Run Strategy

Use llama.cpp with these priorities:
1. Prefer Q3 file first for best chance of near full GPU offload
2. Keep context at 1024 initially
3. Maximize GPU layers without crashing

Start command example on Windows:
llama-server.exe -m path_to_q3_or_q4.gguf -c 1024 -ngl 99 -t 4 --host 0.0.0.0 --port 8080

If VRAM overflow happens, reduce ngl gradually: 99, 80, 64, 48.
If CPU usage goes too high, reduce context first, then move from Q4 to Q3 quant.

A strict 20 percent CPU cap cannot be guaranteed for every prompt length and sampling setup,
but Q3 plus low context usually gives the best chance on GTX 1060 3GB.